# Phase 4: Data Cleaning & Transformation

**Project:** Automated Data Quality & ETL Framework  
**Dataset:** Online Retail  
**Objective:** Clean the validated raw dataset, standardize fields, remove invalid records, create useful transformation columns, and save the cleaned output for the next ETL phase.

This notebook continues after:
- Phase 1: Project Planning
- Phase 2: Dataset Understanding / Data Profiling
- Phase 3: Data Validation


## 1. Import Required Libraries

The libraries below are used for data manipulation, file handling, and timestamp creation for the cleaning summary report.

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

## 2. Define Project Paths

Update `project_folder` only if the project folder is moved to a different location.

Expected local folder structure:

```text
Automated-Data-Quality-ETL-Framework/
├── data/
│   └── raw/
│       └── Online Retail.xlsx
├── outputs/
└── reports/
```

In [ ]:
# Main project folder
project_folder = r"C:\Users\chatu\OneDrive\Desktop\Automated-Data-Quality-ETL-Framework"

# Input file path
input_path = os.path.join(project_folder, "data", "raw", "Online Retail.xlsx")

# Output folders
outputs_folder = os.path.join(project_folder, "outputs")
reports_folder = os.path.join(project_folder, "reports")
cleaning_report_folder = os.path.join(reports_folder, "cleaning")

# Create output folders if they do not already exist
os.makedirs(outputs_folder, exist_ok=True)
os.makedirs(cleaning_report_folder, exist_ok=True)

print("Project paths configured successfully.")
print("Input Path:", input_path)
print("Outputs Folder:", outputs_folder)
print("Cleaning Reports Folder:", cleaning_report_folder)

## 3. Load the Raw Dataset

The raw Excel file is loaded from the `data/raw` folder. The dataset is kept in a DataFrame named `df`.

In [ ]:
# Load raw Excel dataset
df = pd.read_excel(input_path)

print("Dataset loaded successfully.")
print("Initial Shape:", df.shape)

df.head()

## 4. Standardize Column Names

Column names are standardized to lowercase to make the rest of the notebook easier to maintain.

Original columns:

```text
InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country
```

After standardization:

```text
invoiceno, stockcode, description, quantity, invoicedate, unitprice, customerid, country
```

In [ ]:
# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print("Standardized Columns:")
print(df.columns.tolist())

## 5. Remove Duplicate Records

Duplicate rows can distort revenue, order count, and customer-level analysis. This step removes exact duplicate records.

In [ ]:
# Remove exact duplicate rows
initial_rows = len(df)

df = df.drop_duplicates()

duplicates_removed = initial_rows - len(df)

print("Duplicates Removed:", duplicates_removed)
print("Shape After Duplicate Removal:", df.shape)

## 6. Review and Handle Missing Values

Critical fields are required for valid transaction analysis. Rows missing these values are removed.

Critical columns:
- `invoiceno`
- `stockcode`
- `description`
- `quantity`
- `invoicedate`
- `unitprice`
- `customerid`

In [ ]:
# Check missing values before cleaning
missing_before = df.isnull().sum()

print("Missing Values Before Cleaning:")
print(missing_before[missing_before > 0])

In [ ]:
# Remove rows where critical fields are missing
critical_columns = [
    "invoiceno",
    "stockcode",
    "description",
    "quantity",
    "invoicedate",
    "unitprice",
    "customerid"
]

rows_before_missing_cleanup = len(df)

df = df.dropna(subset=critical_columns)

missing_rows_removed = rows_before_missing_cleanup - len(df)

print("Rows Removed Due To Critical Missing Values:", missing_rows_removed)
print("Shape After Missing Value Cleanup:", df.shape)

## 7. Fix Data Types

This step converts columns into the correct formats:
- `invoicedate` → datetime
- `customerid` → string identifier
- `quantity` → numeric
- `unitprice` → numeric

In [ ]:
# Convert invoice date to datetime
df["invoicedate"] = pd.to_datetime(df["invoicedate"], errors="coerce")

# Convert customer ID to string after safely handling numeric values
df["customerid"] = df["customerid"].astype("Int64").astype(str)

# Convert transaction quantity and unit price to numeric
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unitprice"] = pd.to_numeric(df["unitprice"], errors="coerce")

print("Data types fixed successfully.")
print(df.dtypes)

## 8. Remove Invalid Business Records

For sales analysis, records with non-positive quantity or unit price are invalid.

This step removes:
- Quantity less than or equal to zero
- Unit price less than or equal to zero
- Rows where numeric/date conversion failed

In [ ]:
# Remove invalid business records
rows_before_invalid_cleanup = len(df)

# Remove rows where conversion created null values
df = df.dropna(subset=["invoicedate", "quantity", "unitprice"])

# Keep only valid sales records
df = df[df["quantity"] > 0]
df = df[df["unitprice"] > 0]

invalid_rows_removed = rows_before_invalid_cleanup - len(df)

print("Invalid Rows Removed:", invalid_rows_removed)
print("Shape After Invalid Record Removal:", df.shape)

## 9. Create Transformation Columns

These columns make the dataset ready for analytics and reporting.

New columns:
- `total_amount`
- `invoice_year`
- `invoice_month`
- `invoice_day`
- `invoice_weekday`

In [ ]:
# Create sales amount column
df["total_amount"] = df["quantity"] * df["unitprice"]

# Extract date parts for analysis
df["invoice_year"] = df["invoicedate"].dt.year
df["invoice_month"] = df["invoicedate"].dt.month
df["invoice_day"] = df["invoicedate"].dt.day
df["invoice_weekday"] = df["invoicedate"].dt.day_name()

print("Transformation columns created successfully.")

df.head()

## 10. Standardize Text Fields

Text standardization improves grouping consistency during analysis.

- `description` is converted to uppercase.
- `country` is converted to title case.

In [ ]:
# Standardize product description and country names
df["description"] = df["description"].str.strip().str.upper()
df["country"] = df["country"].str.strip().str.title()

print("Text fields standardized successfully.")

## 11. Final Quality Check

This final checkpoint confirms that the cleaned dataset is ready for the next phase.

In [ ]:
print("Final Dataset Shape:", df.shape)

print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

print("\nDuplicate Rows After Cleaning:")
print(df.duplicated().sum())

print("\nInvalid Quantity Records:")
print((df["quantity"] <= 0).sum())

print("\nInvalid Unit Price Records:")
print((df["unitprice"] <= 0).sum())

print("\nCleaned Dataset Preview:")
df.head()

## 12. Save Cleaned Dataset

The cleaned dataset is saved in the `outputs` folder for use in the next phases.

In [ ]:
# Save cleaned dataset
cleaned_file_path = os.path.join(outputs_folder, "cleaned_online_retail_data.csv")

df.to_csv(cleaned_file_path, index=False)

print("Cleaned dataset saved successfully.")
print("File Path:", cleaned_file_path)

## 13. Save Cleaning Summary Report

A cleaning summary is saved under `reports/cleaning`. This gives a quick audit trail of what happened during Phase 4.

In [ ]:
# Create cleaning summary report
summary = pd.DataFrame({
    "Phase": ["Phase 4 - Data Cleaning & Transformation"],
    "Run Date": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
    "Final Rows": [len(df)],
    "Final Columns": [len(df.columns)],
    "Duplicates Removed": [duplicates_removed],
    "Rows Removed Due To Critical Missing Values": [missing_rows_removed],
    "Invalid Rows Removed": [invalid_rows_removed],
    "Cleaned Output File": [cleaned_file_path]
})

summary_path = os.path.join(cleaning_report_folder, "phase_4_cleaning_summary.csv")

summary.to_csv(summary_path, index=False)

print("Cleaning summary saved successfully.")
print("File Path:", summary_path)

summary

## Phase 4 Completion Notes

Phase 4 is complete when both files are created:

```text
outputs/cleaned_online_retail_data.csv
reports/cleaning/phase_4_cleaning_summary.csv
```

The cleaned dataset is now ready for **Phase 5: Feature Engineering**.